In [1]:
! pip install google-api-python-client google-auth

  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.6 MB 1.3 MB/s eta 0:00:13
   - -------------------------------------- 0.5/15.6 MB 7.3 MB/s eta 0:00:03
   --- ------------------------------------ 1.4/15.6 MB 12.4 MB/s eta 0:00:02
   ------ --------------------------------- 2.6/15.6 MB 16.4 MB/s eta 0:00:01
   ---------- ----------------------------- 4.0/15.6 MB 19.9 MB/s eta 0:00:01
   -------------- ------------------------- 5.6/15.6 MB 22.4 MB/s eta 0:00:01
   ------------------ --------------------- 7.2/15.6 MB 24.2 MB/s eta 0:00:01
   ---------------------- ----------------- 8.8/15.6 MB 25.7 MB/s eta 0:00:01
   -------------------------- ------------- 10.3/15.6 MB 28.4 MB/s eta 0:00:01
   ------------------------------ --------- 11.8/15.6 MB 32.7 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 32.7 MB/s e


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Authenticate

In [44]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

SHARED_FILE = 'log.csv'
SERVICE_ACCOUNT_FILE = "driveapiproject-496206-ec015bdb01b0.json"

SCOPES = ["https://www.googleapis.com/auth/drive"]

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

service = build("drive", "v3", credentials=creds)

# View the shared folder and its content

In [38]:
results = service.files().list(
    pageSize=20, 
    fields="files(id, name, mimeType, size)"
).execute()

for file in results.get("files", []):
    name = file.get("name")
    file_type = file.get("mimeType")
    # File size is returned as a string in bytes, or None for folders
    size = file.get("size", "0") 
    
    print(f"Name: {name} | Type: {file_type} | Size: {size} bytes")


Name: log.csv | Type: text/csv | Size: 16 bytes
Name: data_txt | Type: application/vnd.google-apps.folder | Size: 0 bytes
Name: fake_complinace.txt | Type: text/plain | Size: 3739 bytes
Name: sleep-benefits.txt | Type: text/plain | Size: 3944 bytes


# To add data to an existing file

### 1. Find the file id

In [45]:
results = service.files().list(
    q=f"name='{SHARED_FILE}' and trashed=false",
    fields="files(id, name, parents)"
).execute()

files = results.get("files", [])

if not files:
    raise FileNotFoundError(f"{SHARED_FILE} not found or not shared with the service account.")

file_id = files[0]["id"]

print("File ID:", file_id)

File ID: 1wcuAG4NE4QU91o5DgduBCaeOLXU8N_2B


### 2. Download the CSV

In [46]:
from googleapiclient.http import MediaIoBaseDownload
import io

request = service.files().get_media(fileId=file_id)

fh = io.BytesIO()
downloader = MediaIoBaseDownload(fh, request)

done = False
while not done:
    status, done = downloader.next_chunk()

print(fh.getbuffer().nbytes)
fh.seek(0)

16


0

### 3. Append data to file

In [47]:
import pandas as pd

df = pd.read_csv(fh)

print(df)

new_row = pd.DataFrame({
    "name": ["Zena"],
    "age": [25]
})

df = pd.concat([df, new_row], ignore_index=True)
df

  name  age
0  ash   30


,name,age
0,ash,30
1,Zena,25


### 4. Upload the file to Google Drive

In [48]:
import io
from googleapiclient.http import MediaIoBaseUpload

buffer = io.BytesIO()
df.to_csv(buffer, index=False)
buffer.seek(0)    

media = MediaIoBaseUpload(
    buffer,
    mimetype="text/csv",
    resumable=True
)

service.files().update(
    fileId=file_id,
    media_body=media
).execute()

print("CSV updated successfully.")

CSV updated successfully.


# OPTIONAL: A function that does all of the above

In [49]:
import io
import pandas as pd

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaIoBaseUpload,
)

def append_to_google_drive_csv(
    SERVICE_ACCOUNT_FILE: str,
    shared_file_name: str,
    data
):
    """
    Append rows to an existing CSV stored in Google Drive.

    Parameters
    ----------
    SERVICE_ACCOUNT_FILE : str
        Path to the service account JSON.

    shared_file_name : str
        Name of the CSV in Google Drive.

    data : dict or list[dict]
        Row(s) to append.
    """

    # Authenticate
    SCOPES = ["https://www.googleapis.com/auth/drive"]

    creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )

    service = build("drive", "v3", credentials=creds)

    # Find the file
    results = service.files().list(
        q=f"name='{shared_file_name}' and trashed=false",
        fields="files(id,name)"
    ).execute()

    files = results.get("files", [])

    if not files:
        raise FileNotFoundError(
            f"{shared_file_name} not found in Google Drive."
        )

    file_id = files[0]["id"]

    # Download
    request = service.files().get_media(fileId=file_id)

    buffer = io.BytesIO()

    downloader = MediaIoBaseDownload(buffer, request)

    done = False
    while not done:
        _, done = downloader.next_chunk()

    buffer.seek(0)

    # Read CSV
    df = pd.read_csv(buffer)

    # Convert input to DataFrame
    if isinstance(data, dict):
        new_df = pd.DataFrame([data])

    elif isinstance(data, list):
        new_df = pd.DataFrame(data)

    else:
        raise ValueError(
            "data must be a dictionary or a list of dictionaries."
        )

    # Append rows
    df = pd.concat([df, new_df], ignore_index=True)

    # Convert back to CSV
    output = io.BytesIO()
    df.to_csv(output, index=False)
    output.seek(0)

    media = MediaIoBaseUpload(
        output,
        mimetype="text/csv",
        resumable=True
    )

    # Update existing file
    service.files().update(
        fileId=file_id,
        media_body=media
    ).execute()

    print(f"Successfully appended {len(new_df)} row(s) to {shared_file_name}.")

In [50]:
append_to_google_drive_csv(
    SERVICE_ACCOUNT_FILE=SERVICE_ACCOUNT_FILE,
    shared_file_name=SHARED_FILE,
    data={
        "name": "John",
        "age": 32
    }
)

Successfully appended 1 row(s) to log.csv.
